## ASHRAE 140-2023

Section 9: CASE CE420 - Economizer with ODB Limit control

In [1]:
import os

# Archived notebook: case files and weather paths are relative to the test folder
if os.path.basename(os.getcwd()) == "old_notebooks":
    os.chdir("..")
import opensimula as osm
import pandas as pd
import numpy as np

sim = osm.Simulation()
pro = sim.new_project("pro")
pro.read_json("cases/case_CE420.json")

Reading project data from dictonary
Reading completed.
Checking project: Case CE420
Checking completed.
Calculating solar direct shadows ...
Simulating Case CE420: ...


100%|██████████| 8760/8760 [00:20<00:00, 429.34step/s, n_iter=5] 


In [2]:
system = pro.component("system")
sys_df=system.variable_dataframe()
sys_df["time_frac"] = sys_df["F_load"]/sys_df["efficiency_degradation"]
sys_df["power_ve"] = sys_df["time_frac"]*930
sys_df["power_comp"] = sys_df["power"]-sys_df["indoor_fan_power"]-sys_df["power_ve"] 
sys_df["Q_total"] = sys_df["Q_sensible"]+sys_df["Q_latent"]
sys_df["COP2"] = sys_df["Q_total"] / (sys_df["power_comp"] + sys_df["power_ve"])

df_sum = sys_df.resample("YE", on='date').sum()
excel = pd.DataFrame()
excel["Total_consumption"] = [df_sum["power"].iloc[0]/1000]
excel["Compressor_consumption"] = [df_sum["power_comp"].iloc[0]/1000]
excel["Outdoor_fan_consumption"] = [df_sum["power_ve"].iloc[0]/1000]
excel["Indoor_fan_consumption"] = [df_sum["indoor_fan_power"].iloc[0]/1000]
excel["Q_total"] = [df_sum["Q_total"].iloc[0]/1000]
excel["Q_sensible"] = [df_sum["Q_sensible"].iloc[0]/1000]
excel["Q_latent"] = [df_sum["Q_latent"].iloc[0]/1000]
excel["COP2_mean"] = df_sum["Q_total"].iloc[0]/(df_sum["power_comp"].iloc[0]+df_sum["power_ve"].iloc[0])

df_space_mean = pro.component("spaces_1").variable_dataframe(frequency="YE",value="mean")
excel["T_space_mean"] = [df_space_mean["temperature"].iloc[0]]
excel["w_space_mean"] = [df_space_mean["abs_humidity"].iloc[0]/1000]
excel["HR_space_mean"] = [df_space_mean["rel_humidity"].iloc[0]]

excel.to_clipboard()
excel

,Total_consumption,Compressor_consumption,Outdoor_fan_consumption,Indoor_fan_consumption,Q_total,Q_sensible,Q_latent,COP2_mean,T_space_mean,w_space_mean,HR_space_mean
0,32914.099228,19938.271783,2095.907445,10879.92,70715.144306,49874.050374,20841.093933,3.209339,24.060601,0.009318,48.642082


In [3]:
excel_max = pd.DataFrame()
excel_max["max_power"] = [sys_df["power"].max()]
excel_max["max_power_day"] = [pro.dates()[sys_df['power'].idxmax()].strftime("%d-%b")]
excel_max["max_power_hour"] = [int(pro.dates()[sys_df['power'].idxmax()].strftime("%H"))+1]
excel_max["max_Q_sen"] = [sys_df["Q_sensible"].max()]
excel_max["max_Q_sen_day"] = [pro.dates()[sys_df['Q_sensible'].idxmax()].strftime("%d-%b")]
excel_max["max_Q_sen_hour"] = [int(pro.dates()[sys_df['Q_sensible'].idxmax()].strftime("%H"))+1]
excel_max["max_Q_lat"] = [sys_df["Q_latent"].max()]
excel_max["max_Q_lat_day"] = [pro.dates()[sys_df['Q_latent'].idxmax()].strftime("%d-%b")]
excel_max["max_Q_lat_hour"] = [int(pro.dates()[sys_df['Q_latent'].idxmax()].strftime("%H"))+1]
excel_max["max_Q_tot"] = [sys_df["Q_total"].max()]
excel_max["max_Q_tot_day"] = [pro.dates()[sys_df['Q_total'].idxmax()].strftime("%d-%b")]
excel_max["max_Q_tot_hour"] = [int(pro.dates()[sys_df['Q_total'].idxmax()].strftime("%H"))+1]

excel_max.to_clipboard()
excel_max

,max_power,max_power_day,max_power_hour,max_Q_sen,max_Q_sen_day,max_Q_sen_hour,max_Q_lat,max_Q_lat_day,max_Q_lat_hour,max_Q_tot,max_Q_tot_day,max_Q_tot_hour
0,11745.081043,20-Jul,15,23302.081466,20-Jul,15,9717.045434,03-Sep,15,31844.220049,20-Jul,15


In [4]:
excel_max_2 = pd.DataFrame()
excel_max_2["max_COP2"] = [sys_df["COP2"].max()]
excel_max_2["max_COP2_day"] = [pro.dates()[sys_df['COP2'].idxmax()].strftime("%d-%b")]
excel_max_2["max_COP2_hour"] = [int(pro.dates()[sys_df['COP2'].idxmax()].strftime("%H"))+1]
excel_max_2["min_COP2"] = [sys_df["COP2"].min()]
excel_max_2["min_COP2_day"] = [pro.dates()[sys_df['COP2'].idxmin()].strftime("%d-%b")]
excel_max_2["min_COP2_hour"] = [int(pro.dates()[sys_df['COP2'].idxmin()].strftime("%H"))+1]
space_df = pro.component("spaces_1").variable_dataframe()
excel_max_2["max_T_spa"] = [space_df["temperature"].max()]
excel_max_2["max_T_spa_day"] = [pro.dates()[space_df['temperature'].idxmax()].strftime("%d-%b")]
excel_max_2["max_T_spa_hour"] = [int(pro.dates()[space_df['temperature'].idxmax()].strftime("%H"))+1]
excel_max_2["min_T_spa"] = [space_df["temperature"].min()]
excel_max_2["min_T_spa_day"] = [pro.dates()[space_df['temperature'].idxmin()].strftime("%d-%b")]
excel_max_2["min_T_spa_hour"] = [int(pro.dates()[space_df['temperature'].idxmin()].strftime("%H"))+1]
excel_max_2["max_w_spa"] = [space_df["abs_humidity"].max()/1000]
excel_max_2["max_w_spa_day"] = [pro.dates()[space_df['abs_humidity'].idxmax()].strftime("%d-%b")]
excel_max_2["max_w_spa_hour"] = [int(pro.dates()[space_df['abs_humidity'].idxmax()].strftime("%H"))+1]
excel_max_2["min_w_spa"] = [space_df["abs_humidity"].min()/1000]
excel_max_2["min_w_spa_day"] = [pro.dates()[space_df['abs_humidity'].idxmin()].strftime("%d-%b")]
excel_max_2["min_w_spa_hour"] = [int(pro.dates()[space_df['abs_humidity'].idxmin()].strftime("%H"))+1]
excel_max_2["max_hr_spa"] = [space_df["rel_humidity"].max()]
excel_max_2["max_hr_spa_day"] = [pro.dates()[space_df['rel_humidity'].idxmax()].strftime("%d-%b")]
excel_max_2["max_hr_spa_hour"] = [int(pro.dates()[space_df['rel_humidity'].idxmax()].strftime("%H"))+1]
excel_max_2["min_hr_spa"] = [space_df["rel_humidity"].min()]
excel_max_2["min_hr_spa_day"] = [pro.dates()[space_df['rel_humidity'].idxmin()].strftime("%d-%b")]
excel_max_2["min_hr_spa_hour"] = [int(pro.dates()[space_df['rel_humidity'].idxmin()].strftime("%H"))+1]

excel_max_2.to_clipboard()
excel_max_2


,max_COP2,max_COP2_day,max_COP2_hour,min_COP2,min_COP2_day,min_COP2_hour,max_T_spa,max_T_spa_day,max_T_spa_hour,min_T_spa,...,max_w_spa_hour,min_w_spa,min_w_spa_day,min_w_spa_hour,max_hr_spa,max_hr_spa_day,max_hr_spa_hour,min_hr_spa,min_hr_spa_day,min_hr_spa_hour
0,3.782193,21-May,15,2.798879,13-Jun,17,25.075337,03-Sep,15,7.466731,...,3,0.001862,11-Jan,2,70.560047,17-Apr,4,13.316305,06-Nov,6
